# チュートリアル 10: Microsoft Foundry エージェント向け MCP 認証

## エンタープライズ AI にとってなぜこれが重要か

エンタープライズ AI エージェントを構築する際、**セキュリティと適切な認証は交渉の余地がありません**。エージェントには以下が必要です:
- 内部 API とデータベースに安全にアクセス
- 認証情報を公開せずにユーザーに代わって行動
- SaaS ツール (GitHub、Salesforce、M365) と適切に統合
- コンプライアンス (SOC2、HIPAA など) のための監査証跡を維持

**Model Context Protocol (MCP)** は、AI エージェントが外部ツールを呼び出すための標準化された方法を提供します。しかし、エンタープライズでは、API を単に呼び出すことはできません — 適切な認証が必要です。

**このチュートリアルでは、4つの認証パターンを使用して Foundry エージェントを MCP サーバーに安全に接続する方法を示します。**

---

## アーキテクチャの概要

```
┌─────────────────────────────────────────────────────────────────────────────┐
│  Microsoft Foundry MCP Authentication Architecture                          │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  ┌──────────────────┐                                                       │
│  │   User/Client    │                                                       │
│  └────────┬─────────┘                                                       │
│           │                                                                 │
│           ▼                                                                 │
│  ┌──────────────────────────────────────────────────────────────────────┐  │
│  │  Microsoft Foundry Project                                           │  │
│  │  ┌─────────────────────────────────────────────────────────────────┐ │  │
│  │  │  Agent Identity                                                 │ │  │
│  │  │  • 共有プロジェクト ID (未公開エージェント)                      │ │  │
│  │  │  • 個別エージェント ID (公開エージェント)                       │ │  │
│  │  └─────────────────────────────────────────────────────────────────┘ │  │
│  │                                                                      │  │
│  │  ┌─────────────────────────────────────────────────────────────────┐ │  │
│  │  │  Project Connections (認証情報ストア)                           │ │  │
│  │  │  • API Keys, PAT Tokens                                         │ │  │
│  │  │  • OAuth Client Credentials                                     │ │  │
│  │  │  • Agentic Identity Token Config                                │ │  │
│  │  └─────────────────────────────────────────────────────────────────┘ │  │
│  │                                                                      │  │
│  │  ┌─────────────────────────────────────────────────────────────────┐ │  │
│  │  │  Foundry Agent Service                                          │ │  │
│  │  │  • MCP ツールでエージェントを作成                                │ │  │
│  │  │  • 承認ワークフローを管理                                        │ │  │
│  │  │  • OAuth 同意フローを処理                                        │ │  │
│  │  └─────────────────────────────────────────────────────────────────┘ │  │
│  └──────────────────────────────────────────────────────────────────────┘  │
│           │                                                                 │
│           │ Auth Method Selection                                           │
│           ▼                                                                 │
│  ┌───────────────────────────────────────────────────────────────────────┐ │
│  │                     MCP Server                                        │ │
│  │  ┌─────────────┐ ┌─────────────┐ ┌─────────────┐ ┌─────────────────┐ │ │
│  │  │ Unauth      │ │ Key-based   │ │ Agentic ID  │ │ OAuth Passthru  │ │ │
│  │  │ (Public)    │ │ (API Key)   │ │ (MI Token)  │ │ (User Context)  │ │ │
│  │  └─────────────┘ └─────────────┘ └─────────────┘ └─────────────────┘ │ │
│  └───────────────────────────────────────────────────────────────────────┘ │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

## 主要なドキュメントリンク

| リソース | 説明 | URL |
|----------|-------------|-----|
| **MCP Authentication** | Foundry での MCP の公式認証パターン | [learn.microsoft.com](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/mcp-authentication?view=foundry) |
| **Agent Identity Concepts** | Entra ID でのエージェント ID の理解 | [learn.microsoft.com](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/agent-identity?view=foundry) |
| **Azure AI Projects SDK** | エージェント向け Python SDK サンプル | [GitHub](https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/ai/azure-ai-projects/samples) |
| **Azure AI Agents SDK** | MCP サンプル付きエージェント専用 SDK | [GitHub](https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/ai/azure-ai-agents/samples) |
| **Microsoft Foundry Portal** | エージェントの作成と管理 | [ai.azure.com](https://ai.azure.com) |
| **Entra Agent Identities** | Entra でエージェント ID を管理 | [entra.microsoft.com](https://entra.microsoft.com/#view/Microsoft_AAD_RegisteredApps/AllAgents.MenuView) |

## パート 1: Microsoft Foundry でのエージェント ID の理解

### エンタープライズの課題: 「このエージェントは誰?」

AI エージェントが会議室を予約したり、発注を承認したり、顧客データにアクセスしたりすることを想像してください。セキュリティチームは次のように尋ねます: *「エージェントが行ったことと人間が行ったことをどのように監査しますか?」*

従来のサービスプリンシパルは、ボットと人間を区別しません。**Agent Identity** がこれを解決します。

### Agent Identity とは?

**Agent Identity** は、AI エージェント専用に設計された Microsoft Entra ID の特殊なサービスプリンシパルです。AI エージェントに組織内の独自の「社員 ID バッジ」を与えるようなものです。

| 機能 | エンタープライズにとってなぜ重要か |
|-----------|------------------------------|
| **識別された操作** | 監査担当者はログをフィルタリングして「エージェントが何をしたか」vs.「人間が何をしたか」を確認できる |
| **適切なサイズのアクセス** | エージェント固有の権限を付与 (例:「読み取りはできるが削除はできない」) |
| **セキュリティ制御** | エージェントが自己に管理者アクセスを付与したり、権限をエスカレートしたりするのを防ぐ |
| **スケーラブルな管理** | ディレクトリを乱雑にせずに数百のエージェントをスピンアップ/ダウン |

### 共有 vs 個別 Agent Identity

| ID タイプ | 使用する場合 | エンタープライズの例 |
|--------------|-------------|--------------------| 
| **共有プロジェクト ID** | 開発/テストフェーズ | 「HR Assistant」プロジェクトのすべてのエージェントが開発中に1つの ID を共有 |
| **個別エージェント ID** | 本番デプロイ | 「Sales Copilot」は CRM データへのアクセスのみを持つ独自の ID を取得 |

**エンタープライズの推奨事項**: 本番環境では常に**個別 ID**を使用してください。これにより次のことが可能になります:
- エージェントごとの監査証跡 (「どのエージェントがこの顧客レコードにアクセスしたか?」)
- 細かい権限の失効 (他のエージェントに影響を与えずに1つのエージェントを無効化)
- コンプライアンスに優しい関心事の分離

### Agent Identity Blueprint

**Agent Identity Blueprint** は、エージェント ID がどのように作成されるかを管理するテンプレートです:

- **タイプ分類**: エージェントを分類 (例: 「Contoso Sales Agent」)
- **ID 作成権限**: ID 管理のための OAuth 認証情報を定義
- **ランタイム認証**: エージェント実行中にアクセストークンを取得するために使用

### エージェント ID の検索

**共有プロジェクト ID の場合:**
1. [Azure Portal](https://portal.azure.com) に移動
2. Foundry プロジェクトに移動
3. Overview ペインで **JSON View** をクリック
4. `agentIdentityId` 値をコピー

**個別エージェント ID (公開後):**
1. エージェントアプリケーションリソースに移動
2. Overview ペインで **JSON View** をクリック
3. 一意の `agentIdentityId` をコピー

## パート 2: MCP 認証方法

### 適切な認証方法の選択: 実際のシナリオ

Microsoft Foundry は MCP サーバー向けに**4つの主要な認証方法**をサポートしています。選択方法は次のとおりです:

| 方法 | エンタープライズシナリオ | ユーザー操作 | 例 |
|--------|---------------------|------------------|----------|
| **Unauthenticated** | パブリックドキュメント、オープン API | なし | エージェントがパブリック Azure ドキュメント、天気 API を読む |
| **Key-based** | 内部 API、SaaS 統合 | なし | エージェントが GitHub Enterprise、Jira、Confluence をクエリ |
| **Agentic Identity** | Azure ネイティブリソース | なし | エージェントが Azure Blob Storage から読み取り、Cosmos DB をクエリ |
| **OAuth Passthrough** | ユーザー固有のアクセス | はい (同意) | エージェントが*ユーザーとして*メールを送信、*自分の* OneDrive にアクセス |

### 実際のエンタープライズユースケース

| ユースケース | 認証方法 | 理由 |
|----------|-------------|-----|
| **IT ヘルプデスクエージェント**が ServiceNow チケットをクエリ | Key-based | ServiceNow API キーが安全に保存 |
| **営業アシスタント**が会社 CRM を読む | Agentic Identity | エージェントが独自の CRM 権限を持つ |
| **パーソナルスケジューラー**が会議を予約 | OAuth Passthrough | エージェントが Outlook で*ユーザーとして*行動 |
| **コードアシスタント**が Azure API 仕様を検索 | Unauthenticated | パブリックドキュメント |

### 判断ツリー: MCP 認証方法の選択

```
MCP サーバーは認証を必要としますか?
│
├─ NO  → Unauthenticated を使用
│
└─ YES → 各ユーザーは独自の ID/コンテキストが必要ですか?
         │
         ├─ YES → OAuth Identity Passthrough を使用
         │        (ユーザーがサインインし、エージェントが代わりに行動)
         │
         └─ NO  → リソースは Azure サービスですか?
                  │
                  ├─ YES → Agentic Identity を使用
                  │        (エージェント ID に RBAC を割り当て)
                  │
                  └─ NO  → Key-based を使用
                           (API キーを Project Connection に保存)
```

## Azure AI API の理解: Assistants vs Responses

コードに入る前に、Microsoft Foundry で利用可能な**2つの API パラダイム**を理解することが重要です。このチュートリアルでは、新しい **Responses API** を使用しますが、古い **Assistants API** パターンを使用するコードに遭遇する可能性があります。

### API の進化

| 機能 | Assistants API (Preview) | Responses API (New) |
|---------|-------------------------|---------------------|
| **ステータス** | Preview (レガシーパターン) | 一般提供 |
| **状態管理** | Threads + Messages + Runs | `previous_response_id` 経由の会話 |
| **コードパターン** | `client.beta.threads.create()` | `openai_client.responses.create()` |
| **SDK** | `azure-ai-agents` (`McpTool`) | `azure-ai-projects` (`MCPTool`) |
| **MCP サポート** | はい | 承認ワークフロー付きではい |

### Threads vs Conversations: 主な違い

**Assistants API (スレッドベース):**
```python
# 1. スレッド (会話コンテナ) を作成
thread = client.beta.threads.create()

# 2. スレッドにメッセージを追加
client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content="What's the weather?"
)

# 3. スレッドでアシスタントを実行
run = client.beta.threads.runs.create(
    thread_id=thread.id,
    assistant_id=assistant.id
)

# 4. 完了をポーリング
while run.status != "completed":
    run = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)

# 5. メッセージを取得
messages = client.beta.threads.messages.list(thread_id=thread.id)
```

**Responses API (会話ベース):**
```python
# 1. 初期応答を作成 (会話を開始)
response = openai_client.responses.create(
    model="gpt-4o",
    input="What's the weather?",
    tools=[mcp_tool]
)

# 2. previous_response_id で会話を続ける
follow_up = openai_client.responses.create(
    model="gpt-4o",
    input="What about tomorrow?",
    previous_response_id=response.id  # 以前のコンテキストにリンク
)

# 3. 出力に直接アクセス
print(response.output_text)
```

### マルチユーザーアプリケーション設計

複数のユーザーにサービスを提供するエンタープライズアプリケーションを構築する際は、次を考慮してください:

| 側面 | Assistants API | Responses API |
|--------|----------------|---------------|
| **ユーザーの分離** | ユーザーごとに個別の `thread_id` を作成 | ユーザーごとに個別の `previous_response_id` チェーンを使用 |
| **セッションストレージ** | ユーザーセッション/DB に `thread_id` を保存 | ユーザーセッション/DB に最後の `response.id` を保存 |
| **クリーンアップ** | 完了時にスレッドを削除 | レスポンスは自動で期限切れ (デフォルト30日) |
| **同時実行性** | 実行中はスレッドがロック | ロックなし、async フレンドリー |

### エンタープライズアーキテクチャパターン

```
┌─────────────────────────────────────────────────────────────┐
│                     Your Application                        │
├─────────────────────────────────────────────────────────────┤
│  User Session Storage (Redis/Database)                      │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐       │
│  │ User A       │  │ User B       │  │ User C       │       │
│  │ response_id: │  │ response_id: │  │ response_id: │       │
│  │ "resp_abc"   │  │ "resp_xyz"   │  │ "resp_123"   │       │
│  └──────────────┘  └──────────────┘  └──────────────┘       │
└─────────────────────────────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│              Microsoft Foundry (Responses API)               │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  responses.create(previous_response_id=user_resp_id) │   │
│  │  → 各ユーザーが個別の会話チェーンを維持               │   │
│  └─────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────┘
```

### このチュートリアルの主なポイント

1. **Responses API を使用** - モダンな GA アプローチでよりシンプルなコード
2. **`previous_response_id`** - レスポンスをチェーンして会話コンテキストを維持  
3. **`response.output_text`** - モデルからの集約されたテキスト出力
4. **MCP Tool Approvals** - Responses API は MCP ツール呼び出しの承認ワークフローをサポート
5. **スレッド管理なし** - スレッドを作成/ポーリング/管理する必要なし

### SDK リファレンス

| パッケージ | インポートパターン | API スタイル |
|---------|----------------|--------|
| `azure-ai-projects>=2.0.0b1` | `from azure.ai.projects.models import MCPTool` | Responses API |
| `azure-ai-agents>=1.2.0b3` | `from azure.ai.agents.models import McpTool` | Assistants API |

> **注意:** 両方の SDK を一緒にインストールできますが、ツールクラスの名前は似ているが大文字小文字が異なるため (`MCPTool` vs `McpTool`)、インポート文に注意してください。

## パート 3: セットアップとインストール

Microsoft Foundry エージェントと MCP を使用するために必要なパッケージをインストールしましょう。

In [ ]:
# Install required packages
# azure-ai-projects: Foundry プロジェクトとエージェントのメイン SDK
# azure-ai-agents: MCP サポート付きエージェント固有機能
# azure-identity: Azure 認証 (DefaultAzureCredential)

!pip install "azure-ai-projects>=2.0.0b1" "azure-ai-agents>=1.2.0b3" azure-identity python-dotenv --quiet

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)

# =============================================================================
# Configuration
# =============================================================================

# Microsoft Foundry Project Configuration
# Foundry プロジェクトの Overview ページから取得
PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")

# プロジェクトの「Models + endpoints」タブからモデルデプロイ名
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")

# Optional: MCP Project Connection ID (for key-based auth)
MCP_PROJECT_CONNECTION_ID = os.getenv("MCP_PROJECT_CONNECTION_ID", "")

# =============================================================================
# Your Custom MCP Server (from Tutorial 16 - ACA Deployment)
# =============================================================================

# Direct ACA endpoint (from Tutorial 16)
MCP_SERVER_URL = os.getenv("MCP_SERVER_URL", "")
MCP_SERVER_NAME = os.getenv("MCP_SERVER_NAME", "travel-mcp-server")
MCP_BACKEND_URL = os.getenv("MCP_BACKEND_URL", "")

# =============================================================================
# APIM Integration (from Tutorial 18)
# =============================================================================

APIM_NAME = os.getenv("APIM_NAME", "")
APIM_RESOURCE_GROUP = os.getenv("APIM_RESOURCE_GROUP", "")
APIM_GATEWAY_URL = os.getenv("APIM_GATEWAY_URL", "")
APIM_SUBSCRIPTION_KEY = os.getenv("APIM_SUBSCRIPTION_KEY", "")

# Construct APIM-proxied MCP URL (recommended for production)
APIM_MCP_URL = f"{APIM_GATEWAY_URL}/travel-mcp/mcp" if APIM_GATEWAY_URL else ""

# =============================================================================
# Validation
# =============================================================================

def validate_config():
    """Validate required configuration."""
    print("📋 Configuration Status:")
    print("=" * 60)
    
    config_ok = True
    
    # Foundry Configuration
    print("\n🔹 Microsoft Foundry Configuration:")
    if PROJECT_ENDPOINT:
        print(f"  ✅ PROJECT_ENDPOINT: {PROJECT_ENDPOINT[:50]}...")
    else:
        print("  ❌ PROJECT_ENDPOINT: NOT SET")
        config_ok = False
    
    print(f"  ✅ MODEL_DEPLOYMENT_NAME: {MODEL_DEPLOYMENT_NAME}")
    
    if MCP_PROJECT_CONNECTION_ID:
        print(f"  ✅ MCP_PROJECT_CONNECTION_ID: {MCP_PROJECT_CONNECTION_ID[:30]}...")
    else:
        print("  ⚪ MCP_PROJECT_CONNECTION_ID: NOT SET (optional)")
    
    # Custom MCP Server (ACA)
    print("\n🔹 Custom MCP Server (Azure Container Apps):")
    if MCP_SERVER_URL:
        print(f"  ✅ MCP_SERVER_URL: {MCP_SERVER_URL}")
    else:
        print("  ⚪ MCP_SERVER_URL: NOT SET (see Tutorial 16)")
    
    if MCP_BACKEND_URL:
        print(f"  ✅ MCP_BACKEND_URL: {MCP_BACKEND_URL}")
    else:
        print("  ⚪ MCP_BACKEND_URL: NOT SET")
    
    # APIM Configuration
    print("\n🔹 API Management Integration:")
    if APIM_GATEWAY_URL:
        print(f"  ✅ APIM_GATEWAY_URL: {APIM_GATEWAY_URL}")
    else:
        print("  ⚪ APIM_GATEWAY_URL: NOT SET (see Tutorial 18)")
    
    if APIM_SUBSCRIPTION_KEY:
        print(f"  ✅ APIM_SUBSCRIPTION_KEY: {APIM_SUBSCRIPTION_KEY[:8]}...")
    else:
        print("  ⚪ APIM_SUBSCRIPTION_KEY: NOT SET")
    
    if APIM_MCP_URL:
        print(f"  ✅ APIM_MCP_URL: {APIM_MCP_URL}")
    
    print("\n" + "=" * 60)
    
    if not config_ok:
        print("\n📝 Required - Add to your .env file:")
        print('   AZURE_AI_PROJECT_ENDPOINT=https://your-project.api.azureml.ms')
        print('   AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4o')
    
    return config_ok

CONFIG_OK = validate_config()

## パート 4: 非認証 MCP Server

### 使用する場合: パブリック知識ソース

非認証 MCP サーバーは、認証情報を必要としない**公開されている情報**に最適です。

**エンタープライズユースケース:**
- 📖 **ドキュメントルックアップ**: エージェントが公式 Azure/AWS/GCP ドキュメントを使用して質問に回答
- 📊 **パブリックデータ**: 天気、株価、為替レート
- 🔍 **オープンソース探索**: エージェントがパブリック GitHub リポジトリを読む

**MCP Server の例 (パブリック):**

| サーバー | URL | エージェントができること |
|--------|-----|------------------------|
| Azure API Specs | `https://gitmcp.io/Azure/azure-rest-api-specs` | Azure REST API 定義を調べる |
| Microsoft Learn | `https://learn.microsoft.com/api/mcp` | Microsoft ドキュメントを検索 |

> 💡 **ヒント**: パブリックサーバーの場合でも、最初は `require_approval="always"` を使用して、完全に信頼する前にどのツールが呼び出されているかを確認してください。

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, Tool
from openai.types.responses.response_input_param import McpApprovalResponse, ResponseInputParam

def demo_unauthenticated_mcp():
    """
    非認証 (パブリック) MCP サーバーの使用を実演します。
    
    この例は、認証なしで公開アクセス可能な
    Azure REST API 仕様用の GitHub MCP サーバーに接続します。
    """
    
    # MCP ツールを作成 - 認証不要
    mcp_tool = MCPTool(
        server_label="azure_api_specs",  # この MCP サーバーの一意のラベル
        server_url="https://gitmcp.io/Azure/azure-rest-api-specs",  # パブリック MCP エンドポイント
        require_approval="always",  # オプション: "always", "never", または特定のツール
    )
    
    print(f"✅ Created MCP Tool:")
    print(f"   Label: {mcp_tool.server_label}")
    print(f"   URL: {mcp_tool.server_url}")
    print(f"   Approval: {mcp_tool.require_approval}")
    
    return mcp_tool

# Run the demo
mcp_tool_public = demo_unauthenticated_mcp()

## パート 5: Project Connection を使用したキーベース認証

### 使用する場合: SaaS API と内部サービス

ほとんどのエンタープライズ統合には API キーが必要です — GitHub、Jira、Salesforce、ServiceNow など。**Project Connections** により、これらを Azure に安全に保存できます。

**エンタープライズユースケース:**

| サービス | エージェントができること |
|---------|------------------|
| **GitHub Enterprise** | コード検索、issue 作成、PR レビュー |
| **Jira/ServiceNow** | チケット作成、ステータス更新、バックログクエリ |
| **Salesforce** | 顧客データの検索、商談の更新 |
| **Confluence** | 内部 wiki の検索、ランブックの取得 |

### Project Connection の設定 (ステップバイステップ)

1. [Microsoft Foundry Portal](https://ai.azure.com) に移動
2. プロジェクトに移動 → **Management** → **Connected resources**
3. **+ New connection** → **Custom keys** をクリック
4. 設定:
   - **Name**: `github-enterprise-connection`
   - **Key**: `Authorization`
   - **Value**: `Bearer ghp_xxxxxxxxxxxx` (PAT トークン)

### `MCP_PROJECT_CONNECTION_ID` の検索

`MCP_PROJECT_CONNECTION_ID` は Project Connection の **Azure Resource ID** です。MCPTool 設定で接続を参照するためにこれが必要です。

**方法 1: Foundry Portal から (最も簡単)**
1. [ai.azure.com](https://ai.azure.com) → Your Project → **Management** → **Connected resources** に移動
2. 接続名をクリック
3. **Resource ID** を探すか、URL からコピー (以下の形式)

**方法 2: Azure Portal から**
1. [portal.azure.com](https://portal.azure.com) に移動
2. AI Foundry リソース → **Projects** → Your Project に移動
3. **Settings** → **Connections** で、接続をクリック
4. **JSON View** をクリックして `id` フィールドをコピー

**方法 3: Azure CLI を使用**
```bash
# Foundry プロジェクトのすべての接続をリスト
az resource list \
  --resource-group <your-rg> \
  --resource-type "Microsoft.CognitiveServices/accounts/projects/connections" \
  --query "[].{name:name, id:id}" -o table
```

**Resource ID 形式:**
```
/subscriptions/<subscription-id>/resourceGroups/<resource-group>/providers/Microsoft.CognitiveServices/accounts/<ai-services-account>/projects/<project-name>/connections/<connection-name>
```

**`.env` エントリの例:**
```bash
MCP_PROJECT_CONNECTION_ID=/subscriptions/12345678-1234-1234-1234-123456789abc/resourceGroups/my-rg/providers/Microsoft.CognitiveServices/accounts/my-ai-services/projects/my-project/connections/github-mcp-connection
```

### なぜ Project Connections?

| アプローチ | 問題 |
|----------|------|
| ❌ ハードコードされた秘密情報 | git にチェックイン、ログに表示、セキュリティ悪夢 |
| ❌ 環境変数 | より良いが、サーバーアクセス権を持つ誰にでも見える |
| ✅ **Project Connections** | 保存時に暗号化、アクセス制御、監査、ローテーション可能 |

> 🔐 **セキュリティベストプラクティス**: Project Connections は裏で Azure Key Vault を使用します。秘密情報はログやエージェントレスポンスに公開されません。

In [ ]:
def create_mcp_tool_with_connection(connection_id: str) -> MCPTool:
    """
    認証に Project Connection を使用する MCP ツールを作成します。
    
    Project Connection は API キー/トークンを安全に保存し、
    エージェントが MCP サーバーを呼び出すときに自動的に注入します。
    
    Args:
        connection_id: Project Connection のリソース ID
                      (Foundry Portal → Settings → Connections で見つかる)
    
    Returns:
        接続で設定された MCPTool
    """
    
    mcp_tool = MCPTool(
        server_label="github-api",
        server_url="https://api.githubcopilot.com/mcp",  # GitHub の MCP エンドポイント
        require_approval="always",
        project_connection_id=connection_id,  # 保存された認証情報への参照
    )
    
    print(f"✅ Created MCP Tool with Project Connection:")
    print(f"   Label: {mcp_tool.server_label}")
    print(f"   URL: {mcp_tool.server_url}")
    print(f"   Connection ID: {connection_id[:40]}...")
    
    return mcp_tool

# Example usage (only if connection ID is configured)
if MCP_PROJECT_CONNECTION_ID:
    mcp_tool_keybased = create_mcp_tool_with_connection(MCP_PROJECT_CONNECTION_ID)
else:
    print("⚪ Skipping key-based demo - MCP_PROJECT_CONNECTION_ID not set")
    print("   Set up a Project Connection in Foundry Portal to enable this feature")

## パート 6: Agentic Identity 認証

### 使用する場合: Azure ネイティブリソース

エージェントが **Azure サービス** (Storage、Cosmos DB、Key Vault、Logic Apps) にアクセスする必要がある場合、Agentic Identity が最もクリーンなアプローチです — 管理する API キーはありません!

**エンタープライズユースケース:**

| シナリオ | Azure リソース | Agentic Identity のメリット |
|----------|---------------|---------------------------|
| **ドキュメントエージェント**が契約を読む | Azure Blob Storage | キーなし — エージェントが独自の ID を使用 |
| **データエージェント**が顧客レコードをクエリ | Azure Cosmos DB | RBAC がエージェントがアクセスできる内容を正確に制御 |
| **ワークフローエージェント**が自動化をトリガー | Azure Logic Apps | エージェントが独自のマネージド ID を持つ |
| **秘密エージェント**が設定を取得 | Azure Key Vault | エージェントごとに細かい秘密アクセス |

### Agentic Identity の仕組み

```
┌─────────────┐    1. トークンをリクエスト    ┌──────────────────┐
│   Foundry   │──────────────────────▶│    Entra ID      │
│   Agent     │                        │   (Agent's MI)   │
└─────────────┘◀──────────────────────└──────────────────┘
       │           2. トークン (スコープ付き)           
       │                                       
       │       3. トークンで MCP を呼び出し          
       ▼                                       
┌────────────────────────────────────────────┐
│  Your MCP Server (on Azure)                │
│  • JWT トークンを検証                      │
│  • RBAC 権限をチェック                     │
│  • 承認されている場合はツールを実行         │
└────────────────────────────────────────────┘
```

### セットアップ: エージェントに Azure リソースへのアクセスを許可

**ステップ 1: エージェント Identity ID を取得** (エージェントのサービスプリンシパル)
```bash
# Azure Portal → Your Foundry Project → JSON View で
# 次を探す: "agentIdentityId": "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"
```

**ステップ 2: エージェント Identity に RBAC を割り当て**
```bash
# 例: エージェントが blob を読めるようにする (削除はできない!)
az role assignment create \
  --assignee <agent-identity-id> \
  --role "Storage Blob Data Reader" \
  --scope /subscriptions/<sub>/resourceGroups/<rg>/providers/Microsoft.Storage/storageAccounts/<account>
```

> 🔐 **最小権限の原則**: 必要な最小限の権限のみを付与してください。可能な場合は「Reader」ロールを使用してください。

**ステップ 3: AgenticIdentityToken で接続を作成**

In [ ]:
# Example: Creating a connection for Agentic Identity (via REST API)
# これは通常、Azure Portal または Azure CLI を介して行われます

AGENTIC_IDENTITY_CONNECTION_EXAMPLE = """
# REST API リクエストで Agentic Identity 接続を作成
# PUT https://management.azure.com/subscriptions/{sub}/resourceGroups/{rg}/
#     providers/Microsoft.CognitiveServices/accounts/{account}/
#     projects/{project}/connections/{connection_name}?api-version=2024-10-01

{
    "name": "storage-mcp-agentic",
    "type": "CognitiveServices/accounts/projects/connections",
    "properties": {
        "authType": "AgenticIdentityToken",  # エージェントの ID を使用
        "group": "ServicesAndApps",
        "category": "RemoteTool",
        "target": "https://your-mcp-server.azurecontainerapps.io/mcp",
        "isSharedToAll": true,
        "audience": "https://storage.azure.com",  # ターゲットサービススコープ
        "Credentials": {},
        "metadata": {
            "ApiType": "Azure"
        }
    }
}
"""

print("📋 Agentic Identity Connection Example:")
print(AGENTIC_IDENTITY_CONNECTION_EXAMPLE)
print("\n💡 Key Points:")
print("   • authType: 'AgenticIdentityToken' が Foundry にエージェントの ID を使用するように指示")
print("   • audience: ターゲットサービスのスコープ URI")
print("   • 認証情報不要 - ID は自動!")

### Azure サービスの一般的な Audience 値

| Azure Service | Audience URI |
|--------------|---------------|
| Azure Storage | `https://storage.azure.com` |
| Azure Logic Apps | `https://logic.azure.com` |
| Microsoft Foundry | `https://ai.azure.com` |
| Microsoft Graph | `https://graph.microsoft.com` |
| Azure Key Vault | `https://vault.azure.net` |
| Azure Cosmos DB | `https://cosmos.azure.com` |

## パート 7: MCP ツールでエージェントを作成

すべてを統合して、MCP 機能を持つ Foundry エージェントを作成しましょう。

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, Tool

def create_agent_with_mcp(
    project_endpoint: str,
    model_name: str,
    mcp_server_url: str,
    mcp_server_label: str,
    project_connection_id: str = None,
):
    """
    MCP ツール機能を持つ Microsoft Foundry エージェントを作成します。
    
    これは Azure SDK サンプルからの推奨パターンを示しています。
    
    Args:
        project_endpoint: Foundry プロジェクトエンドポイント
        model_name: モデルデプロイ名 (例: 'gpt-4o')
        mcp_server_url: MCP サーバーの URL
        mcp_server_label: この MCP サーバーの一意のラベル
        project_connection_id: 認証された MCP 用のオプション接続 ID
    
    Returns:
        tuple: (agent, project_client) - 呼び出し側はクライアントをクリーンアップする必要があります
    """
    
    # MCP ツールを作成
    mcp_tool_kwargs = {
        "server_label": mcp_server_label,
        "server_url": mcp_server_url,
        "require_approval": "always",
    }
    
    if project_connection_id:
        mcp_tool_kwargs["project_connection_id"] = project_connection_id
    
    mcp_tool = MCPTool(**mcp_tool_kwargs)
    tools: list[Tool] = [mcp_tool]
    
    # クライアントとエージェントを作成 (同期 API)
    credential = DefaultAzureCredential()
    project_client = AIProjectClient(
        endpoint=project_endpoint,
        credential=credential
    )
    
    # MCP ツールでエージェントバージョンを作成
    agent = project_client.agents.create_version(
        agent_name="mcp-demo-agent",
        definition=PromptAgentDefinition(
            model=model_name,
            instructions="""あなたは MCP ツールを使用してユーザーを支援できる役立つエージェントです。

利用可能な MCP ツールを使用して:
- 情報を検索
- 外部リソースにアクセス
- 接続されたデータソースを使用して質問に回答

常に使用しているツールとその理由を説明してください。""",
            tools=tools,
        ),
        description="MCP 機能を持つデモエージェント",
    )
    
    print(f"✅ Agent created:")
    print(f"   ID: {agent.id}")
    print(f"   Name: {agent.name}")
    print(f"   Version: {agent.version}")
    
    return agent, project_client

# Example usage (if configured)
if CONFIG_OK:
    print("🚀 Ready to create agent")
    print("\nExample:")
    print("""agent, client = create_agent_with_mcp(
    project_endpoint=PROJECT_ENDPOINT,
    model_name=MODEL_DEPLOYMENT_NAME,
    mcp_server_url="https://gitmcp.io/Azure/azure-rest-api-specs",
    mcp_server_label="azure-api-specs",
)

# クリーンアップを忘れずに:
# client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
# client.close()""")
else:
    print("❌ Configure PROJECT_ENDPOINT first (see Part 3)")

## パート 8: MCP 承認ワークフローの処理

`require_approval` が `"always"` に設定されているか、特定のツールが含まれている場合、エージェントは MCP ツールを実行する前に承認を要求します。

### 承認フロー

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│    Agent     │────▶│   MCP Tool   │────▶│   Approval   │
│   Request    │     │   Invoked    │     │   Required   │
└──────────────┘     └──────────────┘     └──────┬───────┘
                                                  │
                                                  ▼
                                        ┌──────────────────┐
                                        │ mcp_approval_    │
                                        │ request in       │
                                        │ response.output  │
                                        └────────┬─────────┘
                                                 │
                              ┌──────────────────┼──────────────────┐
                              ▼                                     ▼
                    ┌──────────────────┐               ┌──────────────────┐
                    │  User Approves   │               │  User Denies     │
                    │  (approve=True)  │               │  (approve=False) │
                    └────────┬─────────┘               └────────┬─────────┘
                             │                                  │
                             ▼                                  ▼
                    ┌──────────────────┐               ┌──────────────────┐
                    │ Tool Executes    │               │ Agent Notified   │
                    │ Agent Continues  │               │ of Denial        │
                    └──────────────────┘               └──────────────────┘
```

In [ ]:
from openai.types.responses.response_input_param import McpApprovalResponse, ResponseInputParam

def process_mcp_approval_requests(response, auto_approve: bool = False):
    """
    エージェントレスポンスから MCP 承認リクエストを処理します。
    
    Args:
        response: openai_client.responses.create() からのレスポンス
        auto_approve: True の場合、すべてのリクエストを自動承認
                     False の場合、各リクエストでユーザーにプロンプト
    
    Returns:
        エージェントに送り返す承認レスポンスのリスト
    """
    
    approval_responses: ResponseInputParam = []
    
    for item in response.output:
        if item.type == "mcp_approval_request":
            print(f"\n🔔 MCP Approval Request:")
            print(f"   Request ID: {item.id}")
            print(f"   Server: {item.server_label}")
            
            if auto_approve:
                approved = True
                print(f"   ✅ Auto-approved")
            else:
                # 実際のアプリケーションでは、ユーザーにプロンプトします
                user_input = input("   Approve? (y/n): ").lower().strip()
                approved = user_input in ['y', 'yes']
                print(f"   {'✅ Approved' if approved else '❌ Denied'}")
            
            approval_responses.append(
                McpApprovalResponse(
                    type="mcp_approval_response",
                    approve=approved,
                    approval_request_id=item.id,
                )
            )
    
    return approval_responses

# Example usage
print("📋 MCP Approval Handler defined")
print("\nUsage pattern:")
print("""
# 初期リクエストを送信
response = openai_client.responses.create(
    conversation=conversation.id,
    input="Search for Azure REST API information",
    extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
)

# 承認を処理
approvals = process_mcp_approval_requests(response, auto_approve=True)

# 承認で続ける
if approvals:
    response = openai_client.responses.create(
        input=approvals,
        previous_response_id=response.id,
        extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
    )
""")

## パート 9: OAuth Identity Passthrough

### 使用する場合: ユーザーに代わって行動

エージェントが*ユーザーとして*何かを行う必要がある場合があります — 自分のアカウントからメールを送信したり、個人ファイルにアクセスしたり、カレンダーに会議を予約したりします。

**他の認証方法との主な違い:**
- Key-based: エージェントは**共有サービスアカウント**を使用 (例: `support@company.com`)
- OAuth Passthrough: エージェントは**ログインユーザーとして**行動 (例: `john@company.com`)

**エンタープライズユースケース:**

| シナリオ | なぜ OAuth Passthrough? |
|----------|------------------------|
| **パーソナルアシスタント**がメールを送信 | メールはボットアカウントではなくユーザー「から」表示 |
| **会議スケジューラー**が部屋を予約 | ユーザーのカレンダー権限を使用し、委任ルールを尊重 |
| **ファイルファインダー**が OneDrive を検索 | *そのユーザー*が閲覧権限を持つファイルのみにアクセス |
| **コードレビューア**が GitHub PR を作成 | PR は実際の開発者が作成したものとして表示 |

### OAuth Passthrough の仕組み

```
┌──────────────┐    1. ユーザーがエージェントに    ┌─────────────────┐
│     User     │    "Send email to client" を依頼  │   Foundry Agent │
└──────────────┘─────────────────────────────────▶└─────────────────┘
                                                       │
                       2. エージェントが Outlook アクセスが │
                          必要だがトークンがない          ▼
┌──────────────┐    3. "Click here to         ┌─────────────────┐
│     User     │    authorize Outlook"        │  Consent Prompt │
└──────────────┘◀─────────────────────────────└─────────────────┘
       │
       │ 4. ユーザーがサインインして権限を付与
       ▼
┌──────────────────┐     5. トークンが保存     ┌─────────────────┐
│  Microsoft Login │───────────────────────▶│  Foundry Vault  │
└──────────────────┘                        └─────────────────┘
                                                      │
                       6. エージェントがユーザーの      │
                          トークンを使用してメールを送信  ▼
                                              ┌─────────────────┐
                                              │  Outlook (M365) │
                                              │  Email sent as  │
                                              │  john@company   │
                                              └─────────────────┘
```

### OAuth を使用した Microsoft 365 MCP サーバー

Microsoft は M365 サービス用の**事前構築済み MCP サーバー**を提供 — サーバーデプロイ不要!

| MCP Server | Permission Scope | エンタープライズシナリオ |
|------------|-----------------|---------------------|
| Outlook Mail | `McpServers.Mail.All` | エージェントがユーザーとしてメールを下書き/送信 |
| Outlook Calendar | `McpServers.Calendar.All` | エージェントがユーザーの会議をスケジュール |
| Teams | `McpServers.Teams.All` | エージェントが Teams チャネルに投稿 |
| SharePoint/OneDrive | `McpServers.OneDriveSharepoint.All` | エージェントがユーザーのドキュメントを検索/要約 |
| Copilot Search | `McpServers.CopilotMCP.All` | エージェントがエンタープライズコンテンツを検索 |

> ⚠️ **ユーザープライバシーの考慮事項**: OAuth Passthrough では、エージェントはユーザーが見るものを見ます。不必要に機密性の高い個人データにアクセスしないように、プロンプトを慎重に設計してください。

In [ ]:
# Example: Handling OAuth consent flow

def handle_oauth_consent_request(response):
    """
    MCP ツールからの OAuth 同意リクエストを処理します。
    
    OAuth が必要な場合、レスポンスには
    ユーザーがサインインするための URL を含む
    'oauth_consent_request' が含まれます。
    """
    
    for item in response.output:
        if item.type == "oauth_consent_request":
            print(f"\n🔐 OAuth Consent Required:")
            print(f"   Server: {item.server_label}")
            print(f"   Consent URL: {item.consent_url}")
            print(f"\n   Please have the user visit this URL to authorize access.")
            print(f"   After authorization, re-run the agent request.")
            return item.consent_url
    
    return None

print("📋 OAuth Consent Handler defined")
print("\n💡 OAuth Passthrough is ideal for:")
print("   • Personal GitHub repository access")
print("   • Microsoft 365 calendar/email integration")
print("   • User-specific data that shouldn't be shared")

## パート 10: 完全な例 - マルチ MCP エージェント

### シナリオ: 技術ドキュメントアシスタント

開発者が複数のソースから情報を見つけるのを支援するエージェントを構築しましょう — 単一のエージェントが異なる認証方法で複数の MCP サーバーを使用する方法を示します。

**このエージェントができること:**
- 📖 Azure REST API 仕様を検索 (非認証 MCP)
- 📚 Microsoft Learn ドキュメントをクエリ (非認証 MCP)

> 💡 このパターンはエンタープライズで一般的です: 1つのエージェントが複数の知識ソースに接続し、それぞれが独自の認証要件を持ちます。

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, Tool
from openai.types.responses.response_input_param import McpApprovalResponse, ResponseInputParam

def run_agent_with_mcp_demo():
    """
    MCP ツールを使用した Foundry エージェントの完全なデモンストレーション。
    
    この例は以下を示します:
    1. 非認証 MCP サーバーでエージェントを作成
    2. 会話アプローチの使用 (Foundry Portal パターン)
    3. 承認リクエストの処理
    4. レスポンスの処理
    
    注意: エージェントは削除されないため、再利用したり Foundry Portal で表示したりできます。
    """
    
    if not CONFIG_OK:
        print("❌ Please configure PROJECT_ENDPOINT first")
        return
    
    # MCP Tool 1: パブリック Azure API 仕様 (非認証)
    # 注意: server_label は ^[a-zA-Z0-9_]+$ に一致する必要があります (アンダースコアを使用、ハイフンなし)
    mcp_tool_azure = MCPTool(
        server_label="azure_api_specs",
        server_url="https://gitmcp.io/Azure/azure-rest-api-specs",
        require_approval="always",
    )
    
    # MCP Tool 2: Microsoft Learn ドキュメント (非認証)
    mcp_tool_learn = MCPTool(
        server_label="microsoft_learn",
        server_url="https://learn.microsoft.com/api/mcp",
        require_approval="never",
    )
    
    tools: list[Tool] = [mcp_tool_azure, mcp_tool_learn]
    
    credential = DefaultAzureCredential()
    
    # 同期コンテキストマネージャーを使用
    with (
        AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential) as project_client,
        project_client.get_openai_client() as openai_client,
    ):
        # エージェントを作成 (agent_name はハイフン使用可能)
        agent = project_client.agents.create_version(
            agent_name="multi-mcp-demo-agent",
            definition=PromptAgentDefinition(
                model=MODEL_DEPLOYMENT_NAME,
                instructions="""あなたは以下にアクセスできる役立つ技術アシスタントです:

1. Azure REST API 仕様 (azure_api_specs MCP 経由)
2. Microsoft Learn ドキュメント (microsoft_learn MCP 経由)

これらのツールを使用して Azure サービスに関する技術的な質問に答えてください。
常にソースを引用し、使用しているツールを説明してください。""",
                tools=tools,
            ),
            description="複数の MCP サーバーを持つエージェント",
        )
        
        print(f"✅ Agent created: {agent.name} (v{agent.version})")
        print(f"   💡 Agent persists in Foundry Portal - not auto-deleted")
        
        # 会話を作成 (Foundry Portal パターン)
        conversation = openai_client.conversations.create()
        print(f"✅ Conversation created: {conversation.id}")
        
        # 会話アプローチを使用してクエリを送信
        print("\n📝 Sending query...")
        response = openai_client.responses.create(
            conversation=conversation.id,
            input="Azure Cosmos DB とは何ですか? また、REST API を使用してコンテナーを作成するにはどうすればよいですか?",
            extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
        )
        
        # 承認リクエストを処理
        approval_requests = []
        for item in response.output:
            if item.type == "mcp_approval_request":
                print(f"\n🔔 Approval needed for: {item.server_label}")
                approval_requests.append(
                    McpApprovalResponse(
                        type="mcp_approval_response",
                        approve=True,  # デモ用に自動承認
                        approval_request_id=item.id,
                    )
                )
        
        # 承認リクエストがあった場合、会話を続ける
        if approval_requests:
            print(f"✅ Approved {len(approval_requests)} request(s)")
            response = openai_client.responses.create(
                input=approval_requests,
                previous_response_id=response.id,
                extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
            )
        
        # 最終レスポンスを出力 - すべてのテキストを集約する output_text プロパティを使用
        print("\n📄 Agent Response:")
        print("-" * 60)
        print(response.output_text)
        print("-" * 60)
        
        # エージェントは削除されません - 再利用または Foundry Portal で表示可能
        print(f"\n💡 Agent '{agent.name}' (v{agent.version}) persists in your Foundry project.")
        print(f"   View it at: {PROJECT_ENDPOINT.replace('/api/projects/', '/agents/')}")
        
        return agent

# Run the demo
print("🚀 Multi-MCP Agent Demo")
print("="* 60)
if CONFIG_OK:
    print("Ready to run! Execute: run_agent_with_mcp_demo()")
else:
    print("❌ Configure PROJECT_ENDPOINT to run this demo")

In [ ]:
# Actually run the demo if configured
# このセルは MCP ツールを使用した完全なエージェントワークフローを実行します

if CONFIG_OK:
    print("🚀 Running Multi-MCP Agent Demo...")
    print("=" * 60)
    run_agent_with_mcp_demo()
else:
    print("❌ Configure PROJECT_ENDPOINT to run this demo")

## パート 11: カスタム MCP サーバーの統合

### シナリオ: エンタープライズ旅行予約エージェント

旅行予約用のカスタム MCP サーバーを構築し (チュートリアル 16)、Foundry エージェントに使用させたいと考えています。これは、AI エージェントに公開したい**あらゆる内部 API** のパターンです。

**実際のエンタープライズの類似点:**

| あなたの Travel MCP | エンタープライズの同等物 |
|-------------------|----------------------|
| `search_flights` | 内部在庫システムをクエリ |
| `check_hotel_availability` | リソースの空き状況をチェック (部屋、機器、スロット) |
| `convert_currency` | 価格設定のために財務/ERP API を呼び出し |

### デプロイされた MCP サーバー (チュートリアル 16 から)

| ツール | 機能 | エンタープライズパターン |
|------|--------------|-------------------|
| `search_flights` | 利用可能なフライトを検索 | あらゆる検索/クエリ操作 |
| `check_hotel_availability` | 場所/日付でホテルを検索 | リソースの空き状況チェック |
| `convert_currency` | 通貨換算 | 価格設定/計算 API |

### 接続オプション: 開発 vs 本番

| 環境 | エンドポイント | セキュリティモデル |
|-------------|----------|----------------|
| **開発** | 直接 ACA URL | テスト用により簡単 |
| **本番** | APIM Gateway 経由 | レート制限、サブスクリプションキー、ログ記録 |

### APIM を使用したアーキテクチャ

```
┌─────────────────────────────────────────────────────────────────────┐
│  Production Architecture: Foundry Agent → APIM → ACA MCP Server     │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ┌──────────────┐                                                   │
│  │ Foundry Agent│                                                   │
│  │ (with MCP)   │                                                   │
│  └──────┬───────┘                                                   │
│         │                                                           │
│         │ 1. MCP Request + Auth Header                              │
│         ▼                                                           │
│  ┌────────────────────────────────────────────────────────────────┐│
│  │ Azure API Management                                           ││
│  │ ┌────────────────────────────────────────────────────────────┐ ││
│  │ │ Inbound Policies:                                          │ ││
│  │ │ • validate-azure-ad-token (JWT validation)                 │ ││
│  │ │ • rate-limit-by-key                                        │ ││
│  │ │ • set-header (X-User-Id from token)                        │ ││
│  │ └────────────────────────────────────────────────────────────┘ ││
│  └────────────────────────────────────────────────────────────────┘│
│         │                                                           │
│         │ 2. Validated request                                      │
│         ▼                                                           │
│  ┌────────────────────────────────────────────────────────────────┐│
│  │ Azure Container Apps                                           ││
│  │ ┌────────────────────────────────────────────────────────────┐ ││
│  │ │ FastMCP Travel Server                                      │ ││
│  │ │ • search_flights()                                         │ ││
│  │ │ • check_hotel_availability()                               │ ││
│  │ │ • convert_currency()                                       │ ││
│  │ └────────────────────────────────────────────────────────────┘ ││
│  └────────────────────────────────────────────────────────────────┘│
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Part 11a: Create MCPTool pointing to YOUR deployed Travel MCP Server
from azure.ai.projects.models import MCPTool

def create_travel_mcp_tool(use_apim: bool = True) -> MCPTool:
    """
    デプロイされた Travel MCP Server 用の MCPTool を作成します。
    
    Args:
        use_apim: True の場合、APIM 経由でルート (本番推奨)
                  False の場合、ACA エンドポイントに直接接続
    
    Returns:
        Travel MCP Server 用に設定された MCPTool
    """
    
    if use_apim and APIM_MCP_URL:
        server_url = APIM_MCP_URL
        print(f"🔐 Using APIM-proxied endpoint (production mode)")
    elif MCP_SERVER_URL:
        server_url = MCP_SERVER_URL
        print(f"🔓 Using direct ACA endpoint (development mode)")
    else:
        raise ValueError("No MCP server URL configured. Deploy your server first (Tutorial 16)")
    
    mcp_tool = MCPTool(
        server_label="travel-mcp",
        server_url=server_url,
        require_approval="always",  # 予約操作には承認が必要
    )
    
    print(f"✅ Created Travel MCP Tool:")
    print(f"   Label: {mcp_tool.server_label}")
    print(f"   URL: {server_url}")
    print(f"   Tools available: search_flights, check_hotel_availability, convert_currency")
    
    return mcp_tool

# Create the tool if MCP server is configured
if MCP_SERVER_URL or APIM_MCP_URL:
    travel_mcp_tool = create_travel_mcp_tool(use_apim=bool(APIM_MCP_URL))
else:
    print("⚠️ No MCP server configured. Complete Tutorial 16 first to deploy your Travel MCP Server.")

### Part 11b: 本番環境でなぜ APIM?

エンタープライズデプロイでは、通常、内部 API を直接公開しません。**Azure API Management (APIM)** がセキュアなゲートウェイとして機能します。

**なぜこれが重要か:**

| 懸念事項 | APIM なし | APIM あり |
|---------|-------------|----------|
| **レート制限** | 暴走エージェントが API を圧倒する可能性 | エージェント/ユーザーごとに設定可能なスロットリング |
| **監視** | 可視性が限定的 | 完全なリクエスト/レスポンスのログ記録、分析 |
| **セキュリティ** | 直接公開 | WAF、IP フィルタリング、JWT 検証 |
| **バージョニング** | 管理が困難 | バージョンごとに異なるバックエンドにルーティング |
| **コスト制御** | 無制限の使用 | クォータ強制、使用状況追跡 |

**APIM を使用した認証フロー:**
```
Foundry Agent → APIM (validates subscription key) → Your MCP Server (ACA)
```

> **現在の制限**: MCPTool は `Ocp-Apim-Subscription-Key` のようなカスタムヘッダーをネイティブに渡しません。APIM 統合の場合、APIM を次のように設定してください:
> - サブスクリプションキーなしで Foundry の IP 範囲からのトラフィックを許可、または
> - リクエストソースに基づいてキーを注入するポリシーを使用

In [ ]:
# Part 11c: Create Agent with Travel MCP Server (using Foundry Portal pattern)
# Reference: Microsoft Foundry Portal sample code

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool, PromptAgentDefinition, Tool

def create_travel_agent_foundry_pattern(use_apim: bool = False):
    """
    Azure Container Apps にデプロイされた
    カスタム Travel MCP Server を使用するエージェントを作成します。
    
    これは Foundry Portal パターンを使用:
    - AIProjectClient 
    - MCPTool (大文字の 'CP')
    - openai_client.conversations.create() + responses.create()
    
    注意: エージェントは削除されないため、再利用または Foundry Portal で表示できます。
    
    Args:
        use_apim: True の場合、APIM ゲートウェイを使用。False の場合、直接 ACA エンドポイント。
    """
    
    # Get MCP server URL 
    if use_apim and APIM_MCP_URL:
        mcp_url = APIM_MCP_URL
        print("🔐 Using APIM-proxied endpoint")
    elif MCP_SERVER_URL:
        mcp_url = MCP_SERVER_URL
        print("🔓 Using direct ACA endpoint (no APIM)")
    else:
        print("⚠️ No MCP server configured. Complete Tutorial 16 first.")
        return
    
    print("🚀 Creating Travel Agent with Custom MCP Server (Foundry Pattern)...")
    print("=" * 60)
    print(f"   MCP URL: {mcp_url}")
    
    # Initialize MCP tool using azure.ai.projects MCPTool
    # 注意: server_label は ^[a-zA-Z0-9_]+$ に一致する必要があります (アンダースコア可、ハイフンなし)
    mcp_tool = MCPTool(
        server_label="travel_mcp",
        server_url=mcp_url,
        require_approval="never",  # デモ用に自動承認
    )
    
    tools: list[Tool] = [mcp_tool]
    
    credential = DefaultAzureCredential()
    
    with AIProjectClient(
        endpoint=PROJECT_ENDPOINT,
        credential=credential,
    ) as project_client:
        
        # Get OpenAI client for conversations
        openai_client = project_client.get_openai_client()
        
        # Create agent version with MCP tool
        # 注意: agent_name はハイフン使用可能 (server_label とは異なる!)
        agent = project_client.agents.create_version(
            agent_name="travel-mcp-agent",  # ここではハイフン使用可能
            definition=PromptAgentDefinition(
                model=MODEL_DEPLOYMENT_NAME,
                instructions="""あなたはカスタム MCP サーバーを搭載した役立つ旅行アシスタントです。
            
以下のツールにアクセスできます:
- search_flights: 都市間の利用可能なフライトを検索
- check_hotel_availability: 空室のあるホテルを検索  
- convert_currency: 異なる通貨間で換算

ユーザーが旅行を計画するのを支援する際は、これらのツールを使用して正確な情報を提供してください。
常に結果を明確に提示し、次のステップを支援することを提案してください。""",
                tools=tools,
            ),
            description="カスタム ACA ホスト MCP サーバーを使用する旅行アシスタント",
        )
        
        print(f"✅ Agent created: {agent.name} (v{agent.version})")
        print(f"   ID: {agent.id}")
        print(f"   Model: {MODEL_DEPLOYMENT_NAME}")
        print(f"   💡 Agent persists in Foundry Portal - not auto-deleted")
        print("-" * 60)
        
        # Create conversation (Foundry Portal pattern)
        conversation = openai_client.conversations.create()
        print(f"✅ Conversation created: {conversation.id}")
        
        # Test query using conversations approach (from Foundry Portal)
        test_query = "12月20日に Seattle から New York へ2名で行くフライトを検索してください"
        print(f"\n👤 User: {test_query}")
        print("\n🔄 Processing with Travel MCP Server...")
        
        response = openai_client.responses.create(
            conversation=conversation.id,
            input=test_query,
            extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
        )
        
        print("\n🤖 Assistant response:")
        print("-" * 40)
        print(response.output_text)
        
        # Agent is NOT deleted - can be reused or viewed in Foundry Portal
        print(f"\n💡 Agent '{agent.name}' (v{agent.version}) persists in your Foundry project.")
        print(f"   You can continue conversations or view it in the Foundry Portal.")
        
        return agent, conversation.id

# Run the demo - use direct ACA endpoint (APIM requires subscription key)
if MCP_SERVER_URL or APIM_MCP_URL:
    try:
        create_travel_agent_foundry_pattern(use_apim=False)
    except Exception as e:
        print(f"⚠️ Error: {e}")
        import traceback
        traceback.print_exc()
        print("\nThis may occur if:")
        print("- The MCP server is not running or accessible")
        print("- Network connectivity issues")
else:
    print("⚠️ Travel MCP Server not configured. Complete Tutorial 16 first.")

## パート 12: エンタープライズベストプラクティスとセキュリティチェックリスト

### 認証セキュリティ (セキュリティレビュー用)

| 要件 | 実装方法 | 重要な理由 |
|-------------|-----------------|----------------|
| **ハードコードされた秘密情報なし** | すべての認証情報に Project Connections を使用 | git、ログ、またはエージェント出力での秘密情報を防止 |
| **最小権限アクセス** | エージェント ID に最小限の RBAC ロールを付与 | エージェントが侵害された場合の影響範囲を制限 |
| **すべてを監査** | Azure Monitor + Log Analytics を有効化 | SOC2、HIPAA コンプライアンスに必要 |
| **トークンの衛生管理** | Foundry の自動更新に依存 | 長期有効トークンの盗難を防止 |
| **個別 ID** | 本番用にエージェントを公開 | エージェントごとの監査証跡と失効 |

### 承認ワークフロー (コンプライアンス用)

| データの機密性 | 承認設定 | 例 |
|-----------------|------------------|------|
| **読み取り専用パブリックデータ** | `"never"` | パブリックドキュメントの検索 |
| **内部データの読み取り** | `"always"` (最初) | CRM、HR システムのクエリ |
| **書き込み操作** | `"always"` (常に) | チケット作成、メール送信 |
| **財務/PII データ** | `"always"` + 人によるレビュー | 支払い情報、従業員記録へのアクセス |

### 本番環境準備チェックリスト

**ID とアクセス:**
- [ ] 個別 ID で公開されたエージェント (共有プロジェクト ID ではない)
- [ ] セキュリティチームによるレビューと承認済みの RBAC
- [ ] 過剰な権限を持つロールなし (可能な場合は "Owner"、"Contributor" を避ける)

**秘密管理:**
- [ ] すべての API キーが Project Connections にある (コード内にはない)
- [ ] 秘密ローテーションスケジュールが文書化されている
- [ ] 緊急失効手順がテストされている

**監視とコンプライアンス:**
- [ ] Azure Monitor 診断設定が有効
- [ ] 異常な活動のアラート設定 (高エラー率、予期しないツール)
- [ ] ログ保持がコンプライアンス要件を満たす (90日、1年など)

**ユーザーエクスペリエンス:**
- [ ] 承認プロンプトが明確で実行可能
- [ ] OAuth 同意画面が適切なスコープでレビュー済み
- [ ] エラーメッセージが機密情報を漏らさない

## まとめ: 学んだこと

### 4つの MCP 認証パターン (判断ガイド)

| 必要なこと... | この認証を使用 | 例 |
|--------------------|---------------|------|
| パブリック API/ドキュメントへのアクセス | **Unauthenticated** | Azure API 仕様、パブリック wiki |
| 内部/SaaS API の呼び出し | **Key-based** | GitHub Enterprise、Jira、Salesforce |
| Azure リソースへのアクセス | **Agentic Identity** | Blob Storage、Cosmos DB、Key Vault |
| ユーザーとして行動 | **OAuth Passthrough** | ユーザーとしてメールを送信、自分の OneDrive にアクセス |

### エンタープライズアーキテクチャパターン

```
┌─────────────┐     ┌───────────────┐     ┌──────────────┐     ┌─────────────┐
│   User/App  │────▶│ Foundry Agent │────▶│  MCP Server  │────▶│  Backend    │
│             │     │ (with auth)   │     │  (ACA/APIM)  │     │  Services   │
└─────────────┘     └───────────────┘     └──────────────┘     └─────────────┘
                           │                      │
                    Agent Identity          Rate Limiting
                    RBAC Controls           Monitoring
                    Audit Logging           API Versioning
```

### 主要なセキュリティ原則

1. ✅ **秘密情報をハードコードしない** → Project Connections を使用
2. ✅ **最小権限** → エージェント ID に最小限の権限を付与  
3. ✅ **すべてを監査** → コンプライアンスのために Azure Monitor を有効化
4. ✅ **機密性の高い操作を承認** → 書き込み操作に `require_approval` を使用
5. ✅ **本番環境では個別 ID** → 一意の ID のためにエージェントを公開

### 推奨学習パス

| チュートリアル | 学べること |
|----------|-------------------|
| **15**: FastMCP Server Basics | ゼロから独自の MCP サーバーを構築 |
| **16**: Deploy to ACA | Azure で MCP サーバーをホスト |
| **18**: APIM Integration | エンタープライズゲートウェイを追加 (レート制限、監視) |
| **19**: Orchestrating with MCP | MCP ツールを使用したマルチターン会話 |
| **This Tutorial (19b)** | すべてのシナリオのセキュア認証 |

### 公式リソース

| リソース | 説明 |
|----------|-------------|
| [MCP Authentication Docs](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/mcp-authentication?view=foundry) | すべての認証パターンの公式ガイド |
| [Agent Identity Concepts](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/agent-identity?view=foundry) | Entra エージェント ID の詳細 |
| [Azure AI Projects SDK](https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/ai/azure-ai-projects) | サンプル付き Python SDK |
| [Foundry Portal](https://ai.azure.com) | エージェント、接続、デプロイを管理 |